# Clase 3

En este laboratorio construiremos redes neuronales muy simples para clasificación binaria (0/1), usando un dataset descargado desde internet. Veremos:

1. Cómo importar datos desde la web.
2. Qué significa una “red” de 1 capa y 1 neurona lineal (modelo lineal).
3. Qué cambia cuando usamos una capa oculta de 2 neuronas + activaciones (no linealidad).  
4. Cómo influye el optimizador (Adam vs SGD) en el entrenamiento y los resultados.

Nota conceptual: en redes neuronales, “aprender” significa encontrar pesos ``w`` y sesgo ``b`` para que la salida del modelo prediga bien el target.

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, confusion_matrix


# 1. Exportar Base de Datos desde la Web

OpenML es un repositorio abierto donde se pueden descargar datasets por nombre.
Usaremos el dataset "diabetes" (binario: diabetes = 1, no diabetes = 0).

In [6]:
# -------------------------
# 1) Descargar dataset desde internet
# Dataset: "diabetes" (binario: diabetes sí / no)
# -------------------------
X, y = fetch_openml(
    name="diabetes",
    version=1,
    as_frame=True,
    return_X_y=True
)

In [10]:
X

,preg,plas,pres,skin,insu,mass,pedi,age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [11]:
y

0      tested_positive
1      tested_negative
2      tested_positive
3      tested_negative
4      tested_positive
            ...       
763    tested_negative
764    tested_negative
765    tested_negative
766    tested_positive
767    tested_negative
Name: class, Length: 768, dtype: category
Categories (2, object): ['tested_negative', 'tested_positive']

In [13]:
# Convertimos target a 0/1
# Convertimos target a 0/1
y = y.map({"tested_negative": 0, "tested_positive": 1}).astype(int)
print("Shape:", X.shape)
print("Clases:", y.value_counts().to_dict())

Shape: (768, 8)
Clases: {0: 500, 1: 268}


# 2. Muestra de Entrenamiento y Validación

¿Por qué dividir?

* Entrenamiento: el modelo ajusta sus parámetros.

* Validación: medimos desempeño en datos “nuevos” para estimar generalización.

Usamos stratify=y para que la proporción de clases sea similar en ambos conjuntos. Además, en redes neuronales es muy importante escalar variables (estandarizar) para que las magnitudes no dominen el aprendizaje.

In [14]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)

y_train = y_train.values
y_val = y_val.values


In [17]:
print(X_train_sc.shape)
print(X_val_sc.shape)
print(y_train.shape)
print(y_val.shape)

(576, 8)
(192, 8)
(576,)
(192,)


# 3. Modelo 1

**¿Qué es 1 neurona lineal?**

Una neurona lineal calcula:

$z = w^\top x + b$
donde:
* x: vector de features
* w: pesos aprendidos
* b: sesgo

Si no hay activación, la salida es lineal (no acotada). Para clasificación binaria lo típico es usar una sigmoide, pero aquí lo haremos “todo lineal” para entender la base.

**¿Cómo implementarlo en sklearn?**

* Usamos SGDRegressor: un modelo lineal entrenado por descenso por gradiente.

Luego convertimos el score a [0,1] usando sigmoide solo para evaluar con métricas como AUC y LogLoss (aplicamos sigmoide al score).

In [20]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, confusion_matrix

def eval_binary(y_true, y_score, name="model", threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score)
    ll = log_loss(y_true, np.clip(y_score, 1e-6, 1-1e-6))
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"AUC:      {auc:.4f}")
    print(f"LogLoss:  {ll:.4f}")
    print("CM [[TN FP],[FN TP]]:\n", cm)
    return {"name": name, "acc": acc, "auc": auc, "logloss": ll}

# Modelo lineal (1 neurona)
lin_1 = SGDRegressor(
    loss="squared_error",
    max_iter=5000,
    tol=1e-4,
    random_state=42
)

lin_1.fit(X_train_sc, y_train)

# Score lineal (no es probabilidad)
z_val = lin_1.predict(X_val_sc)

# Sigmoide solo para "mapear" a 0-1 y evaluar
p_val = 1 / (1 + np.exp(-z_val))

res_lin = eval_binary(y_val, p_val, name="1 neurona lineal (SGDRegressor)")



=== 1 neurona lineal (SGDRegressor) ===
Accuracy: 0.4375
AUC:      0.8339
LogLoss:  0.6979
CM [[TN FP],[FN TP]]:
 [[ 17 108]
 [  0  67]]


# 4. Modelo 2

**¿Por qué agregar una capa oculta y activación?

Sin activación, apilar capas lineales sigue siendo lineal.
La no linealidad aparece cuando usamos activaciones como:

* **ReLU**:  $$\text{ReLU}(z) = \max(0, z) $$


* **Sigmoide (logistic)**: $$\sigma(z) = \frac{1}{1 + e^{-z}} $$ (salidas tipo probabilidad)

**Aquí construimos:**

* Capa oculta con 2 neuronas: aprende 2 “representaciones” intermedias.

* Activación en oculta: ReLU o Sigmoide.

* Salida: clasificación binaria (en MLPClassifier se maneja internamente).

En sklearn usamos MLPClassifier(hidden_layer_sizes=(2,)).

In [21]:
from sklearn.neural_network import MLPClassifier

def train_mlp(hidden_activation="relu", solver="adam", lr=0.01, max_iter=400):
    mlp = MLPClassifier(
        hidden_layer_sizes=(2,),
        activation=hidden_activation,   # 'relu' o 'logistic'
        solver=solver,                  # 'adam' o 'sgd'
        learning_rate_init=lr,
        alpha=1e-4,
        max_iter=max_iter,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42
    )
    mlp.fit(X_train_sc, y_train)
    p = mlp.predict_proba(X_val_sc)[:, 1]
    return mlp, p

# 2 neuronas + ReLU
mlp_relu_adam, p_relu_adam = train_mlp(hidden_activation="relu", solver="adam")
res_relu_adam = eval_binary(y_val, p_relu_adam, name="MLP(2 neuronas, ReLU) + Adam")

# 2 neuronas + Sigmoide (logistic)
mlp_sig_adam, p_sig_adam = train_mlp(hidden_activation="logistic", solver="adam")
res_sig_adam = eval_binary(y_val, p_sig_adam, name="MLP(2 neuronas, Sigmoide) + Adam")



=== MLP(2 neuronas, ReLU) + Adam ===
Accuracy: 0.7448
AUC:      0.8012
LogLoss:  0.5156
CM [[TN FP],[FN TP]]:
 [[110  15]
 [ 34  33]]

=== MLP(2 neuronas, Sigmoide) + Adam ===
Accuracy: 0.6406
AUC:      0.4338
LogLoss:  0.6813
CM [[TN FP],[FN TP]]:
 [[119   6]
 [ 63   4]]


# 5. Optimización: ADAM vs SGD

**¿Qué es el optimizador?**

Es la regla que actualiza los pesos ``w`` para reducir la función de pérdida.

* **SGD (Stochastic Gradient Descent)**: actualiza con gradientes “directos”, suele requerir más tuning (learning rate, momentum).

* **Adam**: adapta learning rates por parámetro usando momentos (promedios de gradientes). Suele converger más fácil.

En sklearn, MLPClassifier permite comparar con solver="adam" y solver="sgd".

In [22]:
# Misma arquitectura (2 neuronas, ReLU), distinto optimizador
mlp_relu_sgd, p_relu_sgd = train_mlp(hidden_activation="relu", solver="sgd", lr=0.01)
res_relu_sgd = eval_binary(y_val, p_relu_sgd, name="MLP(2 neuronas, ReLU) + SGD")


=== MLP(2 neuronas, ReLU) + SGD ===
Accuracy: 0.6458
AUC:      0.5624
LogLoss:  0.6533
CM [[TN FP],[FN TP]]:
 [[109  16]
 [ 52  15]]


# 6. Comparador de Modelos

Aquí ordenamos los modelos por AUC (y luego Accuracy).

In [23]:
results = pd.DataFrame([
    {"Modelo": res_lin["name"], "Accuracy": res_lin["acc"], "AUC": res_lin["auc"], "LogLoss": res_lin["logloss"]},
    {"Modelo": res_relu_adam["name"], "Accuracy": res_relu_adam["acc"], "AUC": res_relu_adam["auc"], "LogLoss": res_relu_adam["logloss"]},
    {"Modelo": res_sig_adam["name"], "Accuracy": res_sig_adam["acc"], "AUC": res_sig_adam["auc"], "LogLoss": res_sig_adam["logloss"]},
    {"Modelo": res_relu_sgd["name"], "Accuracy": res_relu_sgd["acc"], "AUC": res_relu_sgd["auc"], "LogLoss": res_relu_sgd["logloss"]},
]).sort_values(["AUC", "Accuracy"], ascending=False)

results


,Modelo,Accuracy,AUC,LogLoss
0,1 neurona lineal (SGDRegressor),0.437500,0.833910,0.697925
1,"MLP(2 neuronas, ReLU) + Adam",0.744792,0.801194,0.515576
3,"MLP(2 neuronas, ReLU) + SGD",0.645833,0.562388,0.653260
2,"MLP(2 neuronas, Sigmoide) + Adam",0.640625,0.433791,0.681320
